<a href="https://colab.research.google.com/github/Krispavnn555/NASSCOM_FDP/blob/main/DAY2_U5_Calculus_for_ML_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U5 — Calculus for ML: Lab

**How models learn — by following the gradient downhill** — derivatives · partial derivatives · the gradient · chain rule · backpropagation · Hessian

_Day 2 · Phase B — Mathematical Foundations · Bridge unit (Module 1.3.8)_

#objectives

By the end of this lab you will be able to:

Compute derivatives numerically (finite difference) and symbolically (SymPy)

Take partial derivatives and assemble the gradient of a multivariable function

Apply the chain rule by hand and verify it with SymPy

Run a forward and backward pass through a 2-layer network and derive its gradients

Compute a Hessian, and see how a gradient-descent step uses the gradient

#how to use this lab

Each section has two kinds of cells:

Worked demo cells — run them top to bottom and read the comments to learn the pattern.

LAB EXERCISE cells (marked 🧪) — your turn. Replace each `# YOUR CODE HERE` with working code.

Run cells with **Shift + Enter**. Run the demos before attempting the exercises.

In [1]:
# Core imports for the whole lab
import numpy as np
import sympy as sp

x, y = sp.symbols('x y')      # symbolic variables we'll reuse
sp.init_printing()            # pretty-print symbolic math
np.random.seed(42)
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)

Setup complete. SymPy 1.14.0 | NumPy 2.0.2


#1. Derivatives — the slope of a function

In [2]:
# -----------------------------------------------------------
# 🔹 1A. NUMERICAL DERIVATIVE (finite difference)
# -----------------------------------------------------------

# The derivative is the slope: how much f changes for a tiny step h
def f(x):
    return x**2

h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')

Numerical f'(3): 6.0  (exact = 6)


In [3]:
# -----------------------------------------------------------
# 🔹 1B. SYMBOLIC DERIVATIVE (SymPy)
# -----------------------------------------------------------

expr = x**2
deriv = sp.diff(expr, x)          # differentiate w.r.t. x
print('d/dx (x**2) =', deriv)     # 2*x

# Evaluate the symbolic derivative at x = 3
print('Symbolic f\'(3) =', deriv.subs(x, 3))

d/dx (x**2) = 2*x
Symbolic f'(3) = 6


#### 🧪 LAB EXERCISE 1 — Numerical vs symbolic derivative

For the function `g(x) = x**3 + 2*x`:
1. Compute the **numerical** derivative at `x = 2` using a finite difference.
2. Compute the **symbolic** derivative with `sp.diff` and print it.
3. Evaluate the symbolic derivative at `x = 2` and confirm it matches step 1.

In [5]:
def g(x_val):
    return x_val**3 + 2*x_val

# 1. Numerical derivative at x = 2 (use h = 1e-6)
h_val = 1e-6
numerical_deriv_at_2 = (g(2 + h_val) - g(2)) / h_val
print('Numerical g\'(2):', round(numerical_deriv_at_2, 4))

# 2. Symbolic derivative of x**3 + 2*x
sym_expr = x**3 + 2*x
symbolic_deriv = sp.diff(sym_expr, x)
print('d/dx (x**3 + 2*x) =', symbolic_deriv)

# 3. Evaluate the symbolic derivative at x = 2 and compare
symbolic_deriv_at_2 = symbolic_deriv.subs(x, 2)
print('Symbolic g\'(2) =', symbolic_deriv_at_2)
print('Match (within tolerance)?', np.isclose(numerical_deriv_at_2, float(symbolic_deriv_at_2)))

Numerical g'(2): 14.0
d/dx (x**3 + 2*x) = 3*x**2 + 2
Symbolic g'(2) = 14
Match (within tolerance)? True


#2. Partial derivatives & the gradient

In [6]:
# -----------------------------------------------------------
# 🔹 2A. PARTIAL DERIVATIVES
# -----------------------------------------------------------

f2 = x**2 + 3*x*y + y**2

# A partial derivative differentiates ONE variable, holding others fixed
print('df/dx =', sp.diff(f2, x))     # 2*x + 3*y
print('df/dy =', sp.diff(f2, y))     # 3*x + 2*y

df/dx = 2*x + 3*y
df/dy = 3*x + 2*y


In [7]:
# -----------------------------------------------------------
# 🔹 2B. THE GRADIENT (vector of partials)
# -----------------------------------------------------------

# The gradient stacks every partial derivative into one vector
grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)

# Evaluate the gradient at the point (x=1, y=2)
grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)   # points in the steepest-ascent direction

grad f = [2*x + 3*y, 3*x + 2*y]
grad f at (1, 2) = [8, 7]


#### 🧪 LAB EXERCISE 2 — Compute a gradient

For `h2 = x**2 * y + sp.sin(y)`:
1. Compute `dh/dx` and `dh/dy` with `sp.diff`.
2. Assemble the gradient as a list `[dh/dx, dh/dy]`.
3. Evaluate the gradient at the point `(x=2, y=0)`.

In [8]:
h2 = x**2 * y + sp.sin(y)

# 1. dh/dx and dh/dy
dhdx = sp.diff(h2, x)
dhdy = sp.diff(h2, y)
print('dh/dx =', dhdx)
print('dh/dy =', dhdy)

# 2. Assemble the gradient list
gradient_h2 = [dhdx, dhdy]
print('Gradient h2 =', gradient_h2)

# 3. Evaluate at (x=2, y=0)
gradient_at_2_0 = [g.subs({x: 2, y: 0}) for g in gradient_h2]
print('Gradient h2 at (2, 0) =', gradient_at_2_0)

dh/dx = 2*x*y
dh/dy = x**2 + cos(y)
Gradient h2 = [2*x*y, x**2 + cos(y)]
Gradient h2 at (2, 0) = [0, 5]


#3. The chain rule

In [9]:
# -----------------------------------------------------------
# 🔹 3A. CHAIN RULE BY HAND vs SymPy
# -----------------------------------------------------------

# y = sin(x**2) is a composition: outer = sin(u), inner = u = x**2
# Chain rule:  dy/dx = cos(u) * du/dx = cos(x**2) * 2x
by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)

print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)

By hand : 2*x*cos(x**2)
By SymPy: 2*x*cos(x**2)
Match?   True


In [10]:
# -----------------------------------------------------------
# 🔹 3B. CHAINING THREE FUNCTIONS
# -----------------------------------------------------------

# y = (3x + 1)**4  -> outer^4, inner (3x+1)
expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))   # 12*(3x+1)^3

d/dx (3x+1)^4 = 12*(3*x + 1)**3


#### 🧪 LAB EXERCISE 3 — Chain rule

For `y = sp.exp(x**2 + 1)`:
1. Write the chain-rule result **by hand** as a SymPy expression (outer derivative × inner derivative).
2. Differentiate the same expression with `sp.diff`.
3. Confirm the two match using `sp.simplify(by_hand - by_sympy) == 0`.

In [11]:
# y = exp(x**2 + 1);  inner = x**2 + 1, inner' = 2x, outer' = exp(inner)

# 1. By hand (as a SymPy expression)
inner_deriv = sp.diff(x**2 + 1, x)
outer_deriv_evaluated = sp.exp(x**2 + 1)
by_hand = outer_deriv_evaluated * inner_deriv
print('By hand :', by_hand)

# 2. With sp.diff
by_sympy = sp.diff(sp.exp(x**2 + 1), x)
print('By SymPy:', by_sympy)

# 3. Confirm they match
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)

By hand : 2*x*exp(x**2 + 1)
By SymPy: 2*x*exp(x**2 + 1)
Match?   True


#4. Backpropagation — the chain rule in a network

A 2-layer network is just a composition:  **x → (W1) → ReLU → (W2) → y_hat**. Backprop applies the chain rule from the loss backwards to every weight.

In [12]:
# -----------------------------------------------------------
# 🔹 4A. FORWARD PASS
# -----------------------------------------------------------

# Tiny toy problem: 4 samples, 3 input features, 5 hidden units, 1 output
X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1

z1 = X @ W1                 # linear layer 1
h  = np.maximum(0, z1)      # ReLU activation
y_hat = h @ W2              # linear layer 2 (prediction)
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))

Initial loss: 1.6936


In [13]:
# -----------------------------------------------------------
# 🔹 4B. BACKWARD PASS (gradients via the chain rule)
# -----------------------------------------------------------

# Work backwards from the loss, one link at a time
dy   = 2 * (y_hat - Y) / Y.size      # d loss / d y_hat
dW2  = h.T @ dy                      # d loss / d W2
dh   = dy @ W2.T                     # d loss / d h
dz1  = dh * (z1 > 0)                 # ReLU gradient (1 where z1>0 else 0)
dW1  = X.T @ dz1                     # d loss / d W1

print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')

dW1 shape: (3, 5) (matches W1)
dW2 shape: (5, 1) (matches W2)


In [14]:
# -----------------------------------------------------------
# 🔹 4C. ONE GRADIENT-DESCENT STEP SHOULD LOWER THE LOSS
# -----------------------------------------------------------

lr = 0.1
W1 -= lr * dW1               # step downhill
W2 -= lr * dW2

# Recompute the loss after the update
h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')

Loss before: 1.6936
Loss after : 1.6524 -> should be lower


#### 🧪 LAB EXERCISE 4 — Derive the gradients for a 2-layer network

Fresh weights are set up below. Using the chain rule (reuse the pattern from 4A/4B):
1. Run the **forward pass**: compute `z1`, `h` (ReLU), `y_hat`, and `loss`.
2. Run the **backward pass**: compute `dy`, `dW2`, `dh`, `dz1`, `dW1`.
3. Take one gradient-descent step (`lr = 0.05`) and print the loss before and after.

In [15]:
Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1     # input -> hidden
Wb = np.random.randn(8, 1) * 0.1     # hidden -> output

# 1. Forward pass: z1, h = ReLU(z1), y_hat, loss
z1 = Xb @ Wa
h = np.maximum(0, z1)
y_hat = h @ Wb
loss = ((y_hat - Yb) ** 2).mean()
print(f'Loss before GD: {loss:.4f}')

# 2. Backward pass: dy, dWb, dh, dz1, dWa
dy = 2 * (y_hat - Yb) / Yb.size
dWb = h.T @ dy
dh = dy @ Wb.T
dz1 = dh * (z1 > 0)
dWa = Xb.T @ dz1

# 3. One step with lr = 0.05; print loss before and after
lr = 0.05
Wa -= lr * dWa
Wb -= lr * dWb

# Recompute the loss after the update
h_new = np.maximum(0, Xb @ Wa)
loss_new = ((h_new @ Wb - Yb) ** 2).mean()
print(f'Loss after GD: {loss_new:.4f}')

Loss before GD: 0.8782
Loss after GD: 0.8719


#5. Hessian & gradient descent

In [16]:
# -----------------------------------------------------------
# 🔹 5A. THE HESSIAN (matrix of second derivatives = curvature)
# -----------------------------------------------------------

f5 = x**2 + 3*x*y + y**2
H = sp.hessian(f5, (x, y))
print('Hessian of f:')
sp.pprint(H)        # [[2, 3], [3, 2]]

Hessian of f:
⎡2  3⎤
⎢    ⎥
⎣3  2⎦


In [17]:
# -----------------------------------------------------------
# 🔹 5B. GRADIENT DESCENT ON A SIMPLE FUNCTION
# -----------------------------------------------------------

# Minimise f(x) = (x - 4)**2, whose minimum is at x = 4.
# Update rule:  x <- x - lr * f'(x),  with f'(x) = 2*(x - 4)
xv = 0.0          # starting point
lr = 0.2
for step in range(15):
    grad = 2 * (xv - 4)        # the derivative
    xv = xv - lr * grad        # step against the gradient
print('Converged x:', round(xv, 3), ' (true minimum = 4)')

Converged x: 3.998  (true minimum = 4)


#### 🧪 LAB EXERCISE 5 — Hessian + gradient descent

1. Compute the **Hessian** of `f = x**4 + y**2` with `sp.hessian`.
2. Run **gradient descent** to minimise `f(x) = (x - 7)**2`:
   start at `x = 0`, `lr = 0.1`, 20 steps, using `f'(x) = 2*(x - 7)`. Print the final `x` (should approach 7).

In [18]:
# 1. Hessian of x**4 + y**2
f_hessian = x**4 + y**2
H_f = sp.hessian(f_hessian, (x, y))
print('Hessian of f = x**4 + y**2:')
sp.pprint(H_f)

# 2. Gradient descent to minimise (x - 7)**2
x_gd = 0.0  # starting point
lr_gd = 0.1
for step in range(20):
    grad_gd = 2 * (x_gd - 7)  # the derivative of (x - 7)**2
    x_gd = x_gd - lr_gd * grad_gd  # step against the gradient
print('Final x after gradient descent:', round(x_gd, 3), '(should approach 7)')

Hessian of f = x**4 + y**2:
⎡    2   ⎤
⎢12⋅x   0⎥
⎢        ⎥
⎣  0    2⎦
Final x after gradient descent: 6.919 (should approach 7)


#📘 Summary — Calculus for ML toolkit

| Concept | What it means | Key calls |
| ------- | ------------- | --------- |
| **Derivative** | slope / rate of change | finite difference · `sp.diff` |
| **Partial derivative** | slope along one variable | `sp.diff(f, x)` |
| **Gradient** | vector of all partials → steepest ascent | `[sp.diff(f, v) for v in vars]` |
| **Chain rule** | differentiate compositions | outer' × inner' (SymPy automates) |
| **Backpropagation** | chain rule through a network | forward pass + backward pass |
| **Hessian** | second derivatives → curvature | `sp.hessian(f, vars)` |
| **Gradient descent** | step downhill | `x -= lr * grad` |

**Homework (self-paced):** numerical vs symbolic derivative · compute ∂f/∂x, ∂f/∂y and assemble ∇f · chain-rule derivation + SymPy check · derive gradients for a 2-layer network · compute a Hessian.

**Next:** U6 Probability & Statistics (Part 1). The full optimization machinery — learning rates, SGD vs Adam, convergence — arrives in U13.